In [29]:
import pandas as pd
import numpy as np
import os
import time

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.decomposition import PCA

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import Input

In [30]:
# 📁 Przygotowanie folderów
os.makedirs('models', exist_ok=True)
os.makedirs('logs', exist_ok=True)
os.makedirs('reports', exist_ok=True)

# 📊 Wczytanie i przygotowanie danych
data = pd.read_excel('złączone_dane.xlsx')
data = data.drop('image_id', axis=1)
data = data.drop(columns=[col for col in data.columns if any(x in col for x in ['3_p', '4_p', '5_p'])])

X = data.drop('label', axis=1)
y = data['label']
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [31]:
# 🎯 PCA grupowe
def apply_grouped_pca(X, n_components=1):
    lm_0_cols = [col for col in X.columns if col.startswith('0_point_lm_')]
    lm_1_cols = [col for col in X.columns if col.startswith('1_point_lm_')]
    lm_2_cols = [col for col in X.columns if col.startswith('2_point_lm_')]
    vec_cols = [col for col in X.columns if '_vec_' in col]

    def pca_transform(cols, prefix):
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X[cols])
        pca = PCA(n_components=n_components)
        X_pca = pca.fit_transform(X_scaled)
        return pd.DataFrame(X_pca, columns=[f'{prefix}_pca_{i}' for i in range(n_components)], index=X.index)

    pca_lm_0 = pca_transform(lm_0_cols, '0')
    pca_lm_1 = pca_transform(lm_1_cols, '1')
    pca_lm_2 = pca_transform(lm_2_cols, '2')
    vec_features = X[vec_cols].reset_index(drop=True)

    return pd.concat([pca_lm_0, pca_lm_1, pca_lm_2, vec_features], axis=1)

X_pca = apply_grouped_pca(X)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_pca)

# 🔁 K-Fold
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

acc_list, prec_list, rec_list, f1_list = [], [], [], []

start_time = time.time()

# 🔁 K-Fold Training
for fold, (train_idx, val_idx) in enumerate(cv.split(X_scaled, y_encoded), 1):
    print(f"\n🔁 Fold {fold}")
    
    X_train, X_val = X_scaled[train_idx], X_scaled[val_idx]
    y_train, y_val = y_encoded[train_idx], y_encoded[val_idx]

    model = Sequential([
        Input(shape=(X_scaled.shape[1],)),
        Dense(128, activation='relu'),
        Dropout(0.3),
        Dense(64, activation='tanh'),
        Dropout(0.2),
        Dense(len(np.unique(y_encoded)), activation='softmax')
    ])

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=50,
        callbacks=[early_stop],
        verbose=0
    )

    y_pred = np.argmax(model.predict(X_val), axis=1)

    acc = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, average='macro')
    rec = recall_score(y_val, y_pred, average='macro')
    f1 = f1_score(y_val, y_pred, average='macro')

    print(f"✅ Fold {fold} — Acc: {acc:.4f}, Prec: {prec:.4f}, Rec: {rec:.4f}, F1: {f1:.4f}")

    acc_list.append(acc)
    prec_list.append(prec)
    rec_list.append(rec)
    f1_list.append(f1)

# 📈 Metryki końcowe
avg_acc = np.mean(acc_list)
avg_prec = np.mean(prec_list)
avg_rec = np.mean(rec_list)
avg_f1 = np.mean(f1_list)
training_time = time.time() - start_time

# 🧾 Raport
y_pred_final = np.argmax(model.predict(X_scaled), axis=1)
report = classification_report(y_encoded, y_pred_final, digits=4)


🔁 Fold 1
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
✅ Fold 1 — Acc: 0.9898, Prec: 0.9682, Rec: 0.9731, F1: 0.9697

🔁 Fold 2
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
✅ Fold 2 — Acc: 0.9898, Prec: 0.9783, Rec: 0.9738, F1: 0.9757

🔁 Fold 3
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
✅ Fold 3 — Acc: 0.9902, Prec: 0.9746, Rec: 0.9714, F1: 0.9720

🔁 Fold 4
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
✅ Fold 4 — Acc: 0.9919, Prec: 0.9733, Rec: 0.9735, F1: 0.9719

🔁 Fold 5
74/74 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
✅ Fold 5 — Acc: 0.9902, Prec: 0.9685, Rec: 0.9659, F1: 0.9667
368/368 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step  


In [32]:
# 💾 Zapis modelu i wyników
model.save('models/tf_ssn_model.h5')
np.save('models/scaler.npy', scaler.mean_)  # lub dump
np.save('models/label_encoder_classes.npy', le.classes_)

with open('reports/tf_ssn_report.txt', 'w', encoding='utf-8') as f:
    f.write("Model: TensorFlow Sequential SSN\n")
    f.write("=== Raport klasyfikacji ===\n")
    f.write(report)
    f.write("\n=== Średnie metryki z CV ===\n")
    f.write(f"Accuracy: {avg_acc:.4f}\n")
    f.write(f"Precision (macro): {avg_prec:.4f}\n")
    f.write(f"Recall (macro): {avg_rec:.4f}\n")
    f.write(f"F1 Score (macro): {avg_f1:.4f}\n")
    f.write(f"\nCzas treningu: {training_time:.2f} sekund\n")

with open('logs/tf_ssn_log.txt', 'w', encoding='utf-8') as f:
    f.write(f"Model: TensorFlow Sequential SSN\n")
    f.write(f"Czas treningu: {training_time:.2f} s\n")
    f.write(f"Epoki: {len(history.history['loss'])}\n")
    f.write(f"Parametry: {model.count_params()} total\n")
    f.write(f"Accuracy: {avg_acc:.4f}\n")
    f.write(f"Precision: {avg_prec:.4f}\n")
    f.write(f"Recall: {avg_rec:.4f}\n")
    f.write(f"F1: {avg_f1:.4f}\n")

# 📋 Konsola
print("\n📊 Wyniki końcowe (średnie):")
print('|====================|')
print(f"Accuracy: {avg_acc:.4f}")
print(f"Precision: {avg_prec:.4f}")
print(f"Recall: {avg_rec:.4f}")
print(f"F1: {avg_f1:.4f}")
print(f"Czas treningu: {training_time:.2f} s")
print('|====================|')


📊 Wyniki końcowe (średnie):
|====================|
Accuracy: 0.9904
Precision: 0.9726
Recall: 0.9715
F1: 0.9712
Czas treningu: 161.10 s
|====================|


In [33]:
import pandas as pd
import numpy as np
import os
from joblib import load
from tensorflow.keras.models import load_model
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.preprocessing import LabelEncoder
# === ŚCIEŻKI ===
MODEL_PATH = 'models/tf_ssn_model.h5'
ENCODER_PATH = 'models/label_encoder_classes.npy'
TEST_DATA_PATH = 'test_data.xlsx'

# === FUNKCJE POMOCNICZE ===
def preprocess_features(X: pd.DataFrame, n_components: int = 1) -> pd.DataFrame:
    """
    Przetwarza dane wejściowe:
    - osobne PCA dla każdej grupy punktów,
    - zachowuje cechy wektorowe,
    - skaluje dane przed PCA.
    """
    lm_0_cols = [col for col in X.columns if col.startswith('0_point_lm_')]
    lm_1_cols = [col for col in X.columns if col.startswith('1_point_lm_')]
    lm_2_cols = [col for col in X.columns if col.startswith('2_point_lm_')]
    vec_cols = [col for col in X.columns if '_vec_' in col]

    def pca_transform(cols, prefix):
        if not cols:
            return pd.DataFrame(index=X.index)
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X[cols])
        pca = PCA(n_components=min(n_components, len(cols)))
        X_pca = pca.fit_transform(X_scaled)
        return pd.DataFrame(X_pca, columns=[f'{prefix}_pca_{i}' for i in range(X_pca.shape[1])], index=X.index)

    pca_lm_0 = pca_transform(lm_0_cols, '0')
    pca_lm_1 = pca_transform(lm_1_cols, '1')
    pca_lm_2 = pca_transform(lm_2_cols, '2')
    vec_features = X[vec_cols].reset_index(drop=True)

    return pd.concat([pca_lm_0, pca_lm_1, pca_lm_2, vec_features], axis=1)

# === 1. Wczytaj model i encoder ===
if not os.path.exists(MODEL_PATH) or not os.path.exists(ENCODER_PATH):
    raise FileNotFoundError("Model lub encoder nie został znaleziony.")

model = load_model(MODEL_PATH)

# Wczytaj klasy zakodowane przez np.save
label_classes = np.load(ENCODER_PATH, allow_pickle=True)

# Odtwórz encoder
label_encoder = LabelEncoder()
label_encoder.classes_ = label_classes

# === 2. Wczytaj dane testowe ===
df = pd.read_excel(TEST_DATA_PATH)

# Usuwanie niepotrzebnych kolumn (jak w treningu)
df = df.drop(columns=[col for col in df.columns if any(x in col for x in ['3_p', '4_p', '5_p'])], errors='ignore')
if 'image_id' in df.columns:
    df = df.drop('image_id', axis=1)

# === 3. Preprocessing ===
X_test = preprocess_features(df, n_components=1)

# Skaler całościowy — jak w treningu TF (globalne skalowanie przed siecią)
scaler = StandardScaler()
X_test_scaled = scaler.fit_transform(X_test)  # UWAGA: w praktyce należy zapisać i wczytać scaler z treningu!

# === 4. Predykcja ===
y_pred_proba = model.predict(X_test_scaled)

# === 5. Przykład: Prawdopodobieństwa dla pierwszej próbki ===
first_sample_proba = y_pred_proba[0]
class_labels = label_encoder.inverse_transform(np.arange(len(first_sample_proba)))
results = dict(zip(class_labels, np.round(first_sample_proba, 4)))

# === 6. Wynik ===
print('\n📊 Prawdopodobieństwa klas dla pierwszej próbki:')
for label, prob in results.items():
    print(f"{label}: {prob:.4f}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step

C:\Users\PC2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\decomposition\_pca.py:586: RuntimeWarning: invalid value encountered in divide
  explained_variance_ = (S**2) / (n_samples - 1)
C:\Users\PC2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\decomposition\_pca.py:586: RuntimeWarning: invalid value encountered in divide
  explained_variance_ = (S**2) / (n_samples - 1)
C:\Users\PC2\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\sklearn\decomposition\_pca.py:586: RuntimeWarning: invalid value encountered in divide
  explained_variance_ = (S**2) / (n_samples - 1)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step

📊 Prawdopodobieństwa klas dla pierwszej próbki:
a: 0.0021
a+: 0.0106
b: 0.0017
c: 0.0018
c+: 0.0189
ch: 0.1714
cz: 0.0213
d: 0.0072
e: 0.0015
e+: 0.0527
f: 0.0225
g: 0.0112
h: 0.0515
i: 0.0013
j: 0.0177
k: 0.0154
l: 0.0012
l+: 0.0145
m: 0.0015
n: 0.0025
n+: 0.0206
o: 0.0016
o+: 0.0182
p: 0.0019
r: 0.0028
rz: 0.2041
s: 0.0013
s+: 0.0452
sz: 0.0261
t: 0.0015
u: 0.0017
w: 0.0040
y: 0.0027
z: 0.1253
z+: 0.0383
z-: 0.0761
